# LSTM Model

Multi-task: **primary** = 5-day return (regression, evaluated by R2); **secondary** = up/down direction (classification, evaluated by dir_acc/AUC). Shared bidirectional-attention backbone; the auxiliary direction loss also regularizes the representation.

## Import Libraries

In [1]:
import json
import os

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

## Parameters

In [2]:
DATASET_NAME = "vcb_lb20_h5_f200_dynta_tr70_val15_test15_std"
DATA_DIR = os.path.join("../../train_test_set", DATASET_NAME)

CLIP_VALUE = 10.0
USE_TOP_FEATURES = 80     # best single-stock setting

# --- model (small + regularized) ---
HIDDEN_SIZE = 48
NUM_LAYERS = 1
DROPOUT = 0.4

# --- multi-task ---
DIR_WEIGHT = 0.3          # weight of the auxiliary direction (BCE) loss

# --- training ---
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
EPOCHS = 100
PATIENCE = 12
GRAD_CLIP = 1.0
RANDOM_STATE = 42

L.seed_everything(RANDOM_STATE, workers=True)

Seed set to 42


42

## Load Data

Primary target = scaled 5-day return. Secondary label = up/down, derived from the real (inverse-transformed) return sign.

In [3]:
def load(name):
    return np.load(os.path.join(DATA_DIR, name))

X_train, yr_train = load("X_train.npy"), load("y_train.npy")
X_val, yr_val = load("X_val.npy"), load("y_val.npy")
X_test, yr_test = load("X_test.npy"), load("y_test.npy")

with open(os.path.join(DATA_DIR, "metadata.json")) as f:
    metadata = json.load(f)
target_scaler = joblib.load(os.path.join(DATA_DIR, "target_scaler.pkl"))

if USE_TOP_FEATURES:
    feat_cols = metadata["feature_columns"]
    ranking = pd.read_csv(os.path.join(DATA_DIR, "feature_ranking.csv"))
    ranked = ranking[ranking["feature"].isin(feat_cols)].sort_values("blended_score", ascending=False)
    idx = [feat_cols.index(c) for c in ranked["feature"].head(USE_TOP_FEATURES)]
    X_train, X_val, X_test = X_train[:, :, idx], X_val[:, :, idx], X_test[:, :, idx]
    sel_names = [feat_cols[i] for i in idx]
    n_macro = sum(n.startswith(('economy_', 'bonds_')) for n in sel_names)
    print(f"Subset to {len(idx)} features  (macro:{n_macro} TA/price:{len(idx)-n_macro})")

def to_dir(yr):
    real = target_scaler.inverse_transform(yr.reshape(-1, 1)).ravel()
    return (real > 0).astype(np.float32)

yd_train, yd_val, yd_test = to_dir(yr_train), to_dir(yr_val), to_dir(yr_test)
N_FEATURES = X_train.shape[-1]
print(f"X_train {X_train.shape}  up-rate train={yd_train.mean():.3f} test={yd_test.mean():.3f}")

Subset to 80 features  (macro:13 TA/price:67)
X_train (2926, 20, 80)  up-rate train=0.497 test=0.437


## DataModule

In [4]:
class StockDataModule(L.LightningDataModule):
    def __init__(self, splits, batch_size, clip_value):
        super().__init__()
        self.splits = splits
        self.batch_size = batch_size
        self.clip_value = clip_value

    def _ds(self, key):
        X, yr, yd = self.splits[key]
        X = np.clip(X, -self.clip_value, self.clip_value)
        return TensorDataset(torch.from_numpy(X).float(),
                             torch.from_numpy(yr).float().unsqueeze(-1),
                             torch.from_numpy(yd).float().unsqueeze(-1))

    def setup(self, stage=None):
        self.train_ds, self.val_ds, self.test_ds = self._ds("train"), self._ds("val"), self._ds("test")

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False)


splits = {"train": (X_train, yr_train, yd_train), "val": (X_val, yr_val, yd_val), "test": (X_test, yr_test, yd_test)}
datamodule = StockDataModule(splits, BATCH_SIZE, CLIP_VALUE)

## Model

In [5]:
class MultiTaskLSTM(L.LightningModule):
    """Bidirectional LSTM + attention pooling -> shared context -> return head + direction head.

    Loss = Huber(return) + DIR_WEIGHT * BCE(direction). Return is the primary target.
    """

    def __init__(self, num_features, hidden_size, num_layers, dropout, lr, weight_decay, dir_weight):
        super().__init__()
        self.save_hyperparameters()
        self.lstm = nn.LSTM(num_features, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0, bidirectional=True)
        d = hidden_size * 2
        self.attn = nn.Linear(d, 1)
        self.trunk = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Dropout(dropout))
        self.return_head = nn.Linear(d // 2, 1)
        self.dir_head = nn.Linear(d // 2, 1)
        self.huber = nn.SmoothL1Loss(beta=1.0)

    def forward(self, x):
        out, _ = self.lstm(x)
        w = torch.softmax(self.attn(out), dim=1)
        ctx = (w * out).sum(dim=1)
        h = self.trunk(ctx)
        return self.return_head(h), self.dir_head(h)

    def _step(self, batch, stage):
        x, yr, yd = batch
        ret, logit = self(x)
        loss_ret = self.huber(ret, yr)
        loss_dir = F.binary_cross_entropy_with_logits(logit, yd)
        loss = loss_ret + self.hparams.dir_weight * loss_dir
        self.log(f"{stage}_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
        self.log(f"{stage}_ret", loss_ret, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def training_step(self, b, _):
        return self._step(b, "train")
    def validation_step(self, b, _):
        return self._step(b, "val")
    def test_step(self, b, _):
        return self._step(b, "test")
    def predict_step(self, b, _):
        x = b[0]
        ret, logit = self(x)
        return ret, torch.sigmoid(logit)

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "monitor": "val_loss"}}


model = MultiTaskLSTM(N_FEATURES, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, LEARNING_RATE, WEIGHT_DECAY, DIR_WEIGHT)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

Trainable parameters: 54,771


## Train

In [6]:
early_stop = EarlyStopping(monitor="val_loss", mode="min", patience=PATIENCE)
checkpoint = ModelCheckpoint(dirpath="checkpoints", filename=f"lstm_mt_{DATASET_NAME}",
                             monitor="val_loss", mode="min", save_top_k=1)
logger = CSVLogger(save_dir=".", name="lightning_logs")

trainer = L.Trainer(max_epochs=EPOCHS, accelerator="auto", devices=1, gradient_clip_val=GRAD_CLIP,
                    callbacks=[early_stop, checkpoint], logger=logger, log_every_n_steps=10,
                    enable_progress_bar=True)
trainer.fit(model, datamodule=datamodule)
print(f"Best val_loss = {float(checkpoint.best_model_score):.4f}")

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


You are using a CUDA device ('NVIDIA GeForce RTX 3050 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


D:\GIT\master-thesis\mt_env\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory D:\GIT\master-thesis\src\model\lstm\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

Validation DataLoader 0: 100%|██████████| 10/10 [00:00<00:00, 272.87it/s]

Epoch 12: 100%|██████████| 46/46 [00:00<00:00, 126.25it/s, v_num=8, val_loss=0.982, val_ret=0.643, train_loss=0.291, train_ret=0.162]

Epoch 12: 100%|██████████| 46/46 [00:00<00:00, 125.90it/s, v_num=8, val_loss=0.982, val_ret=0.643, train_loss=0.273, train_ret=0.149]

Epoch 12: 100%|██████████| 46/46 [00:00<00:00, 124.88it/s, v_num=8, val_loss=0.982, val_ret=0.643, train_loss=0.273, train_ret=0.149]


Best val_loss = 0.4789


## Evaluate

In [7]:
best_model = MultiTaskLSTM.load_from_checkpoint(checkpoint.best_model_path)

def report(name, X, yr, yd):
    Xc = np.clip(X, -CLIP_VALUE, CLIP_VALUE)
    loader = DataLoader(TensorDataset(torch.from_numpy(Xc).float()), batch_size=BATCH_SIZE)
    preds = trainer.predict(best_model, loader)
    ret = torch.cat([p[0] for p in preds]).numpy().ravel()
    prob = torch.cat([p[1] for p in preds]).numpy().ravel()
    # return metrics (real units)
    p = target_scaler.inverse_transform(ret.reshape(-1, 1)).ravel()
    t = target_scaler.inverse_transform(yr.reshape(-1, 1)).ravel()
    ss_res = float(np.sum((t - p) ** 2)); ss_tot = float(np.sum((t - t.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot
    rmse = float(np.sqrt(np.mean((p - t) ** 2)))
    corr = float(np.corrcoef(p, t)[0, 1])
    # direction-head metrics
    dpred = (prob > 0.5).astype(int)
    dir_acc = float((dpred == yd).mean())
    auc = float(roc_auc_score(yd, prob)) if len(np.unique(yd)) > 1 else float('nan')
    print(f"{name:5s} | RET: R2={r2:+.4f} RMSE={rmse:.3f} corr={corr:+.3f} | DIR: acc={dir_acc:.3f} AUC={auc:.3f}")
    return p, t

print("primary=return (R2), secondary=direction (acc/AUC)")
report("train", X_train, yr_train, yd_train)
report("val", X_val, yr_val, yd_val)
test_pred, test_true = report("test", X_test, yr_test, yd_test)

primary=return (R2), secondary=direction (acc/AUC)


D:\GIT\master-thesis\mt_env\Lib\site-packages\lightning\fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


D:\GIT\master-thesis\mt_env\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]

...

Predicting DataLoader 0:  80%|████████  | 8/10 [00:00<00:00, 551.41it/s]

Predicting DataLoader 0:  90%|█████████ | 9/10 [00:00<00:00, 512.23it/s]

Predicting DataLoader 0: 100%|██████████| 10/10 [00:00<00:00, 510.48it/s]

Predicting DataLoader 0: 100%|██████████| 10/10 [00:00<00:00, 485.59it/s]


test  | RET: R2=+0.0152 RMSE=3.555 corr=+0.142 | DIR: acc=0.432 AUC=0.513


## Save Model

In [8]:
print(f"Best checkpoint: {checkpoint.best_model_path}")
print(f"Best val_loss:   {float(checkpoint.best_model_score):.4f}")
print(f"CSV logs:        {logger.log_dir}")

Best checkpoint: D:\GIT\master-thesis\src\model\lstm\checkpoints\lstm_mt_vcb_lb20_h5_f200_dynta_tr70_val15_test15_std.ckpt
Best val_loss:   0.4789
CSV logs:        .\lightning_logs\version_8
